In [1]:
from workflow_agent import LlmHandler

llm_handler = LlmHandler(
    llm_h_type="ollama",
    llm_h_params={
        "connection_string": "http://localhost:11434",
        "model_name": "gpt-oss:20b" #"llama3.1:latest" # "gemma3:27b"
    }
)

In [2]:
from workflow_agent import create_function_item

from typing import Type
from pydantic import BaseModel, Field

# --- Example usage ---

# Define mock functions and their associated Pydantic models:

# 1. get_weather function


class GetWeatherInput(BaseModel):
    city: str = Field(..., description="Name of the city for which weather to be extracted.")

class GetWeatherOutput(BaseModel):
    condition: str = Field(..., description="Weather condition in the requested city.")
    temperature: float = Field(..., description="Termperature in C in the requested city.")
    humidity: float = Field(None, description="Name of the city for which weather to be extracted.")

def get_weather(inputs: GetWeatherInput) -> GetWeatherOutput:
    """Get the current weather for a city from weather forcast api."""
    return GetWeatherOutput(
        condition = "Sunny",
        temperature = 20,
        humidity = 0.6
    )

# 2. send_report_email function

class EmailInformationPoint(BaseModel):
    title: str = Field(None, description="Few word description of the information.")
    content: str = Field(..., description="Content of the information.")

class SendReportEmailInput(BaseModel):
    city: str = Field(..., description="Name of the city where report will be send.")
    information: list[EmailInformationPoint]

class SendReportEmailOutput(BaseModel):
    email_sent: bool = Field(..., description="Conformation that email was send successfully.")
    message: str = Field(None, description="Optional comments from the process.")

def send_report_email(inputs: SendReportEmailInput) -> SendReportEmailOutput:
    """Sends a report email with given information points to a city."""
    return SendReportEmailOutput(
        email_sent = True,
        message = "Email sent to city of your choosing!"
    )

# 3. query_database function

class QueryDatabaseInput(BaseModel):
    topic: str = Field(..., description="Topic of a requested piece of information.")
    location: str = Field(None, description="Filter for location name.")
    uid: str = Field(None, description="Filter for unique indentifier of the database item.")

class QueryDatabaseOutput(BaseModel):
    info: str = Field(..., description="Content of the information.")
    uid: str = Field(None, description="Unique indentifier of the database item.")

def query_database(inputs : QueryDatabaseInput) -> QueryDatabaseOutput:
    """Get information from the database with provided filters."""
    return QueryDatabaseOutput(
        info = "Content extracted from the database for your query is ...",
        uid = "0000"
    )

# Create structured items for each function
available_functions = [
    create_function_item(get_weather, GetWeatherInput, GetWeatherOutput),
    create_function_item(send_report_email, SendReportEmailInput, SendReportEmailOutput),
    create_function_item(query_database, QueryDatabaseInput, QueryDatabaseOutput)
]

available_callables = {
    "get_weather" : get_weather,
    "send_report_email" : send_report_email,
    "query_database" : query_database
}

### 0. Initialize Planner and Adaptor

In [3]:
from workflow_agent import WorkflowPlanner
import logging

wp = WorkflowPlanner(
    llm_h = llm_handler,
    available_functions=available_functions,
    loggerLvl = logging.DEBUG)

In [4]:
from workflow_agent import WorkflowAdaptor
from workflow_agent import InputCollector
import logging

wa = WorkflowAdaptor(
    llm_h = llm_handler, 
    input_collector_class = InputCollector,
    available_functions=available_functions,
    loggerLvl = logging.DEBUG)

In [5]:
from workflow_agent import WorkflowRunner

wr = WorkflowRunner(
    available_callables = available_callables, 
    available_functions = available_functions)

### 1. Generating simple workflow using available functions (no input or output model)

In [6]:
task_description = """Query database to find information on birds and get latest weather for Berlin, then send an email there."""

workflow = await wp.generate_workflow(
    task_description=task_description)

workflow

DEBUG:WorkflowPlanner:Attempt: 1


[{'name': 'query_database', 'args': {'topic': 'birds'}},
 {'name': 'get_weather', 'args': {'city': 'Berlin'}},
 {'name': 'send_report_email',
  'args': {'city': 'Berlin',
   'information': [{'title': 'Bird Information',
     'content': 'source: query_database.output.info'},
    {'title': 'Weather', 'content': 'source: get_weather.output.condition'}]}}]

In [7]:
adapted_workflow = await wa.adapt_workflow(
    workflow=workflow)

adapted_workflow

DEBUG:WorkflowAdaptor:step: get_weather adapted successfully on step 1! -------------------------------
DEBUG:WorkflowAdaptor:step: send_report_email adapted successfully on step 1! -------------------------------
DEBUG:InputCollector:og_leaves : {'[0].args.topic': 'birds', '[1].args.city': 'Berlin', '[2].args.city': 'Berlin', '[2].args.information[0].title': 'Bird Information', '[2].args.information[0].content': 'source: query_database.output.info', '[2].args.information[1].title': 'Weather', '[2].args.information[1].content': 'source: get_weather.output.condition'}
DEBUG:InputCollector:mod_leaves : {'[0].id': '1', '[0].args.topic': 'birds', '[1].id': '2', '[1].args.city': 'Berlin', '[2].id': '3', '[2].args.city': 'Berlin', '[2].args.information[0].title': 'Bird Information', '[2].args.information[0].content': '1.output.info', '[2].args.information[1].title': 'Weather', '[2].args.information[1].content': '2.output.condition'}
DEBUG:InputCollector:ic_results : ['literal', 'literal', 'l

[{'id': 1, 'name': 'query_database', 'args': {'topic': 'birds'}},
 {'id': 2, 'name': 'get_weather', 'args': {'city': 'Berlin'}},
 {'id': 3,
  'name': 'send_report_email',
  'args': {'city': 'Berlin',
   'information': [{'title': 'Bird Information', 'content': '1.output.info'},
    {'title': 'Weather', 'content': '2.output.condition'}]}}]

### 2. Generating simple workflow using available functions (no output model)

In [8]:
task_description_a = """Query database to find information on birds and get latest weather for the city, then send an email there."""

class WfInputs(BaseModel):
    city: str = Field(..., description="Name of the city for which weather to be extracted.")

workflow_a = await wp.generate_workflow(
    input_model = WfInputs,
    task_description=task_description_a)

workflow_a

DEBUG:WorkflowPlanner:Attempt: 1


[{'name': 'query_database', 'args': {'topic': 'birds'}},
 {'name': 'get_weather', 'args': {'city': 'source: input_model.output.city'}},
 {'name': 'send_report_email',
  'args': {'city': 'source: input_model.output.city',
   'information': [{'title': 'Birds Info',
     'content': 'source: step1.output.info'},
    {'title': 'Weather', 'content': 'source: step2.output.condition'}]}}]

In [9]:
adapted_workflow_a = await wa.adapt_workflow(
    workflow=workflow_a, 
    input_model = WfInputs)

adapted_workflow_a

DEBUG:WorkflowAdaptor:step: query_database adapted successfully on step 1! -------------------------------
DEBUG:WorkflowAdaptor:step: send_report_email adapted successfully on step 1! -------------------------------
DEBUG:WorkflowAdaptor:step: get_weather adapted successfully on step 1! -------------------------------
DEBUG:InputCollector:og_leaves : {'[0].args.topic': 'birds', '[1].args.city': 'source: input_model.output.city', '[2].args.city': 'source: input_model.output.city', '[2].args.information[0].title': 'Birds Info', '[2].args.information[0].content': 'source: step1.output.info', '[2].args.information[1].title': 'Weather', '[2].args.information[1].content': 'source: step2.output.condition'}
DEBUG:InputCollector:mod_leaves : {'[0].id': '1', '[0].args.topic': 'birds', '[1].id': '2', '[1].args.city': '0.output.city', '[2].id': '3', '[2].args.city': '0.output.city', '[2].args.information[0].title': 'Birds Info', '[2].args.information[0].content': '1.output.info', '[2].args.inform

[{'id': 1, 'name': 'query_database', 'args': {'topic': 'birds'}},
 {'id': 2, 'name': 'get_weather', 'args': {'city': '0.output.city'}},
 {'id': 3,
  'name': 'send_report_email',
  'args': {'city': '0.output.city',
   'information': [{'title': 'Birds Info', 'content': '1.output.info'},
    {'title': 'Weather', 'content': '2.output.condition'}]}}]

### 3. Generating simple workflow using available functions

In [10]:
task_description_b = """Query database to find information on birds and get latest weather for the city, then send an email there."""

class WfInputs(BaseModel):
    city: str = Field(..., description="Name of the city for which weather to be extracted.")

class WfOutputs(BaseModel):
    city: str = Field(..., description="Name of the city for which weather was extracted.")
    information: list[EmailInformationPoint] = Field(default=[], description="Information sent via email.")

workflow_b = await wp.generate_workflow(
    input_model = WfInputs,
    output_model = WfOutputs,
    task_description=task_description_b)

workflow_b

DEBUG:WorkflowPlanner:Attempt: 1


[{'name': 'query_database', 'args': {'topic': 'birds'}},
 {'name': 'get_weather', 'args': {'city': 'source: input_model.output.city'}},
 {'name': 'send_report_email',
  'args': {'city': 'source: input_model.output.city',
   'information': [{'title': 'Birds Information',
     'content': 'source: query_database.output.info'},
    {'title': 'Weather Condition',
     'content': 'source: get_weather.output.condition'}]}}]

In [11]:
adapted_workflow_b = await wa.adapt_workflow(
    workflow=workflow_b, 
    output_model = WfOutputs,
    input_model = WfInputs)

adapted_workflow_b

DEBUG:WorkflowAdaptor:step: send_report_email adapted successfully on step 1! -------------------------------
DEBUG:WorkflowAdaptor:{
  "city": "id_0.output.city",
  "information": [
    {
      "title": "Birds Information",
      "content": "id_1.output.info"
    },
    {
      "title": "Weather Condition",
      "content": "id_2.output.condition"
    }
  ]
}
 Mapping for key 'city' is invalid.
 Function '' not found in state for reference id_1.output.info.
 Mapping for key 'content' is invalid.
 Mapping for array item at index 0 is invalid.
 Function '' not found in state for reference id_2.output.condition.
 Mapping for key 'content' is invalid.
 Mapping for array item at index 1 is invalid.
 Mapping for key 'information' is invalid.
DEBUG:WorkflowAdaptor:step: get_weather adapted successfully on step 1! -------------------------------
DEBUG:WorkflowAdaptor:step: query_database adapted successfully on step 1! -------------------------------
DEBUG:WorkflowAdaptor:step: output_model a

[{'id': 1, 'name': 'query_database', 'args': {'topic': 'birds'}},
 {'id': 2, 'name': 'get_weather', 'args': {'city': '0.output.city'}},
 {'id': 3,
  'name': 'send_report_email',
  'args': {'city': '0.output.city',
   'information': [{'title': 'Birds Information', 'content': '1.output.info'},
    {'title': 'Weather Condition', 'content': '2.output.condition'}]}},
 {'id': '4',
  'name': 'output_model',
  'args': {'city': '0.output.city',
   'information': [{'title': 'Birds Information', 'content': '1.output.info'},
    {'title': 'Weather Condition', 'content': '2.output.condition'}]}}]

In [12]:
test_outputs = wr.run_workflow(
    workflow = adapted_workflow_b, 
    inputs = WfInputs(city = "Berlin"),
    output_model = WfOutputs)

test_outputs.outputs

{'0': WfInputs(city='Berlin'),
 '1': QueryDatabaseOutput(info='Content extracted from the database for your query is ...', uid='0000'),
 '2': GetWeatherOutput(condition='Sunny', temperature=20.0, humidity=0.6),
 '3': SendReportEmailOutput(email_sent=True, message='Email sent to city of your choosing!'),
 '4': WfOutputs(city='Berlin', information=[EmailInformationPoint(title='Birds Information', content='Content extracted from the database for your query is ...'), EmailInformationPoint(title='Weather Condition', content='Sunny')])}